In [ ]:
from pathlib import Path
from html import unescape
import re

import pandas as pd

RAW_DIR = Path('../data/raw')
OUTPUT_PATH = Path('../data/processed/news_clean.csv')

TRUE_PATH = RAW_DIR / 'True.csv'
FAKE_PATH = RAW_DIR / 'Fake.csv'

TRUE_PATH.exists(), FAKE_PATH.exists()

## Cleaning Helpers

For now, helpers stay inside this notebook because you asked to keep Python-related ML work in `.ipynb` files.

In [ ]:
WHITESPACE_RE = re.compile(r'\s+')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')


def clean_text(value):
    if value is None:
        return ''

    text = unescape(str(value))
    text = URL_RE.sub(' ', text)
    text = EMAIL_RE.sub(' ', text)
    text = text.replace('\x00', ' ')
    text = WHITESPACE_RE.sub(' ', text)
    return text.strip()


def build_content(title, body):
    return clean_text(f'{clean_text(title)} {clean_text(body)}')

## Load And Label Raw Data

In [ ]:
real_df = pd.read_csv(TRUE_PATH)
fake_df = pd.read_csv(FAKE_PATH)

real_df['label'] = 'REAL'
fake_df['label'] = 'FAKE'

raw_df = pd.concat([real_df, fake_df], ignore_index=True)
raw_df.shape

In [ ]:
raw_df.head()

## Build Standard Columns

The processed dataset will use only these columns:

- `title`
- `text`
- `content`
- `label`

In [ ]:
df = pd.DataFrame()
df['title'] = raw_df['title'].map(clean_text)
df['text'] = raw_df['text'].map(clean_text)
df['content'] = [build_content(title, text) for title, text in zip(df['title'], df['text'])]
df['label'] = raw_df['label']

df.head()

## Remove Bad Rows

In [ ]:
before_rows = len(df)

df = df.dropna(subset=['content', 'label'])
df = df[df['label'].isin(['REAL', 'FAKE'])]
df = df[df['content'].str.len() >= 80]
df = df.drop_duplicates(subset=['content'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

after_rows = len(df)
before_rows, after_rows, before_rows - after_rows

In [ ]:
df['label'].value_counts()

## Save Processed Dataset

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

OUTPUT_PATH, OUTPUT_PATH.exists()